In [ ]:
import duckdb
import os

WAREHOUSE_DB = r"C:\Users\howar\OneDrive\Documents\BU Mod2\699 pt2\Real Estate\warehouse\realestate_warehouse.duckdb"
GOLD_DB = r"C:\Users\howar\OneDrive\Documents\BU Mod2\699 pt2\Real Estate\gold\realestate_gold.duckdb"

RAW_DEED_FOLDER = r"C:\path\to\deed_files"  # <-- UPDATE

con = duckdb.connect(WAREHOUSE_DB)
con.execute(f"ATTACH '{GOLD_DB}' AS gold")

print("Connected to warehouse + gold")


#LOAD RAW FILES 


con.execute("DROP TABLE IF EXISTS deed_rpi_instruments")
con.execute("DROP TABLE IF EXISTS deed_rp_names")
con.execute("DROP TABLE IF EXISTS deed_rp_legal_desc")

con.execute(f"""
CREATE TABLE deed_rpi_instruments AS
SELECT * FROM read_csv_auto('{RAW_DEED_FOLDER}/RPIInstruments.txt', delim='|', header=True)
""")

con.execute(f"""
CREATE TABLE deed_rp_names AS
SELECT * FROM read_csv_auto('{RAW_DEED_FOLDER}/RPNames.txt', delim='|', header=True)
""")

con.execute(f"""
CREATE TABLE deed_rp_legal_desc AS
SELECT * FROM read_csv_auto('{RAW_DEED_FOLDER}/RPLegalDesc.txt', delim='|', header=True)
""")

print("Raw deed files loaded")


#BUILD CLEAN DEED EVENTS


con.execute("DROP TABLE IF EXISTS deed_transfer_events_clean")

con.execute("""
CREATE TABLE deed_transfer_events_clean AS
SELECT
    "File No" AS file_no,
    "Document Type" AS doc_type,
    CAST("FileDate" AS DATE) AS file_date
FROM deed_rpi_instruments
WHERE "Document Type" IN ('W/D', 'DEED', 'QCD', 'CONVEY', 'TRSALE', 'PROB')
""")

print("Clean deed events created")


#BUILD PARTY AGGREGATIONS
#

con.execute("DROP TABLE IF EXISTS deed_event_parties")

con.execute("""
CREATE TABLE deed_event_parties AS
SELECT
    "File No" AS file_no,
    STRING_AGG(CASE WHEN "Type of Name" = 1 THEN "Name" END, '; ') AS sellers,
    STRING_AGG(CASE WHEN "Type of Name" = 2 THEN "Name" END, '; ') AS buyers
FROM deed_rp_names
GROUP BY "File No"
""")

print("Buyer/seller aggregation complete")


#BUILD LEGAL KEYS (DEEDS)


con.execute("DROP TABLE IF EXISTS deed_transfer_events_with_legal")

con.execute("""
CREATE TABLE deed_transfer_events_with_legal AS
SELECT
    d.file_no,
    d.doc_type,
    d.file_date,
    l."Subdivision",
    l."Block",
    l."Lot",
    CONCAT(
        LOWER(TRIM(l."Subdivision")), '|',
        LOWER(TRIM(l."Block")), '|',
        LOWER(TRIM(l."Lot"))
    ) AS legal_key
FROM deed_transfer_events_clean d
LEFT JOIN deed_rp_legal_desc l
ON d.file_no = l."File No"
""")

print("Legal keys built for deeds")


#PREP GOLD PROPERTY LEGAL KEYS


con.execute("DROP TABLE IF EXISTS gold.property_anchor_with_legal_key")

con.execute("""
CREATE TABLE gold.property_anchor_with_legal_key AS
SELECT
    *,
    CONCAT(
        LOWER(TRIM(subdivision)), '|',
        LOWER(TRIM(block)), '|',
        LOWER(TRIM(lot))
    ) AS legal_key
FROM gold.property_anchor
""")

print("Gold legal keys built")


#MATCH DEEDS TO PROPERTIES


con.execute("DROP TABLE IF EXISTS deed_property_matches")

con.execute("""
CREATE TABLE deed_property_matches AS
SELECT
    d.file_no,
    d.file_date,
    d.doc_type,
    p.acct,
    d.legal_key
FROM deed_transfer_events_with_legal d
INNER JOIN gold.property_anchor_with_legal_key p
ON d.legal_key = p.legal_key
""")

print("Deeds matched to properties")


#BUILD OWNERSHIP HISTORY


con.execute("DROP TABLE IF EXISTS gold.ownership_history")

con.execute("""
CREATE TABLE gold.ownership_history AS
SELECT
    acct,
    file_no,
    file_date AS ownership_start_date,
    LEAD(file_date) OVER (PARTITION BY acct ORDER BY file_date) AS ownership_end_date
FROM deed_property_matches
""")

print("Ownership history created")


#BUILD CURRENT OWNER VIEW


con.execute("DROP VIEW IF EXISTS gold.current_owner")

con.execute("""
CREATE VIEW gold.current_owner AS
SELECT *
FROM gold.ownership_history
WHERE ownership_end_date IS NULL
""")

print("Current owner view created")


#VALIDATION SUMMARY


total_events = con.execute("SELECT COUNT(*) FROM deed_transfer_events_clean").fetchone()[0]
total_matches = con.execute("SELECT COUNT(*) FROM deed_property_matches").fetchone()[0]
total_properties = con.execute("SELECT COUNT(DISTINCT acct) FROM deed_property_matches").fetchone()[0]

print("========== PIPELINE SUMMARY ==========")
print(f"Total deed events: {total_events}")
print(f"Matched events: {total_matches}")
print(f"Matched properties: {total_properties}")